# Smart Packaging and Spoilage Monitoring of White Button Mushrooms
## Deep Learning Visual Quality Classification: ResNet50 vs. Swin Transformer
**Research Project:** Master's Food Science & Computer Vision Research
**Objective:** Classify the Visual Quality Grade (4 Classes: Very Healthy, Healthy, Spoiled, Very Spoiled) based on cap browning and shrinkage defects. Compare a classic CNN (ResNet50) vs. a Vision Transformer (Swin-T) with Grad-CAM explainability.


In [ ]:
import os
import sys
import time
import zipfile
import shutil
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import transforms
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

print('=' * 70)
print('ENVIRONMENT & HARDWARE ACCELERATION CHECK')
print('=' * 70)
print(f'PyTorch Version:    {torch.__version__}')
print(f'CUDA Available:     {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device:      {device}')
if device.type == 'cuda':
    print(f'GPU Device Name:    {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory Total:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
print('=' * 70)


### 1. Dataset Extraction & Directory Resolution
Extracts the preprocessed 12.9 MB dataset archive (mushroom_preprocessed_224.zip) into /kaggle/working/dataset/.


In [ ]:
DATASET_DIR = Path('/kaggle/working/dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# 1. Search for zip in Kaggle input or local directory
zip_candidates = list(Path('/kaggle/input').glob('**/*mushroom*.zip')) + list(Path('.').glob('**/*mushroom*.zip'))

if zip_candidates:
    chosen_zip = zip_candidates[0]
    print(f'Found dataset archive: {chosen_zip}')
    with zipfile.ZipFile(chosen_zip, 'r') as zf:
        zf.extractall(DATASET_DIR)
    print(f'Extracted dataset to: {DATASET_DIR}')
else:
    # Check if files already present in input
    pre_candidates = list(Path('/kaggle/input').glob('**/Control'))
    if pre_candidates:
        DATASET_DIR = pre_candidates[0].parent
        print(f'Using pre-mounted dataset directory: {DATASET_DIR}')
    else:
        print('[Warning] No zip found in /kaggle/input. Using current directory.')
        DATASET_DIR = Path('preprocessed_224').resolve()

# Verify contents
treatments = [d.name for d in DATASET_DIR.iterdir() if d.is_dir() and not d.name.startswith('.')]
print(f'Discovered treatments: {treatments}')
image_count = len(list(DATASET_DIR.glob('*/*/*.*')))
print(f'Total preprocessed images available: {image_count} (Expected: 792)')


### 2. Biological Quality Grade Labeling & PyTorch Dataset
Maps (Treatment, Day) to 4 physiological quality grades to account for packaging spoilage delays.
Ensures zero data leakage by strictly partitioning by **Sample ID (1 to 22)**.


In [ ]:
LABEL_MAPPING = {
    # Class 0 (Very Healthy): Control Day 0 | 1SKEO.2CEO Days 0, 3 | 2SKEO.1CEO Days 0, 3
    ('Control', 0): 0, ('1SKEO.2CEO', 0): 0, ('1SKEO.2CEO', 3): 0,
    ('2SKEO.1CEO', 0): 0, ('2SKEO.1CEO', 3): 0,

    # Class 1 (Healthy): Control Day 3 | 1SKEO.2CEO Days 7, 11 | 2SKEO.1CEO Days 7, 11
    ('Control', 3): 1, ('1SKEO.2CEO', 7): 1, ('1SKEO.2CEO', 11): 1,
    ('2SKEO.1CEO', 7): 1, ('2SKEO.1CEO', 11): 1,

    # Class 2 (Spoiled): Control Days 7, 11 | 1SKEO.2CEO Day 20 | 2SKEO.1CEO Day 20
    ('Control', 7): 2, ('Control', 11): 2,
    ('1SKEO.2CEO', 20): 2, ('2SKEO.1CEO', 20): 2,

    # Class 3 (Very Spoiled): Control Days 20, 30 | 1SKEO.2CEO Day 30 | 2SKEO.1CEO Day 30
    ('Control', 20): 3, ('Control', 30): 3,
    ('1SKEO.2CEO', 30): 3, ('2SKEO.1CEO', 30): 3,
}

CLASS_NAMES = [
    'Class 0: Very Healthy',
    'Class 1: Healthy',
    'Class 2: Spoiled',
    'Class 3: Very Spoiled'
]

import json, re

def get_splits(data_dir):
    split_file = data_dir / 'splits.json'
    if split_file.exists():
        with open(split_file, 'r') as f:
            return json.load(f)
    # Default deterministic partition (seed=42)
    return {
        'train': [2, 3, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 22],
        'val': [8, 9, 18],
        'test': [1, 4, 21]
    }

class MushroomDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split.lower()
        self.transform = transform
        self.splits = get_splits(self.root_dir)
        self.allowed_ids = set(self.splits[self.split]) if self.split != 'all' else set(range(1, 23))
        self.samples = self._load_samples()

    def _load_samples(self):
        samples = []
        pattern = re.compile(r'^(\d+)-([bBsS])\.')
        for img_path in sorted(self.root_dir.glob('*/*/*.*')):
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            t_name = img_path.parent.parent.name
            day = int(img_path.parent.name) if img_path.parent.name.isdigit() else -1
            if (t_name, day) not in LABEL_MAPPING:
                continue
            m = pattern.match(img_path.name)
            if not m:
                continue
            s_id = int(m.group(1))
            if s_id in self.allowed_ids:
                samples.append({
                    'path': str(img_path),
                    'treatment': t_name,
                    'day': day,
                    'sample_id': s_id,
                    'label': LABEL_MAPPING[(t_name, day)]
                })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item['path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, item['label']

# Transforms & Loaders
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = MushroomDataset(DATASET_DIR, split='train', transform=train_tf)
val_ds = MushroomDataset(DATASET_DIR, split='val', transform=eval_tf)
test_ds = MushroomDataset(DATASET_DIR, split='test', transform=eval_tf)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train Samples: {len(train_ds)} (16 IDs) | Val Samples: {len(val_ds)} (3 IDs) | Test Samples: {len(test_ds)} (3 IDs)')


### 3. Model Building & Transfer Learning (ResNet50 vs. Swin-T)
Both pre-trained architectures adapted for 4 Visual Quality Classes with Two-Stage Fine-Tuning support.


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights, swin_t, Swin_T_Weights

def build_resnet50(num_classes=4, pretrained=True):
    weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = resnet50(weights=weights)
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.fc.in_features, num_classes)
    )
    model.model_name = 'resnet50'
    return model

def build_swin_t(num_classes=4, pretrained=True):
    weights = Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
    model = swin_t(weights=weights)
    model.head = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.head.in_features, num_classes)
    )
    model.model_name = 'swin_t'
    return model

def freeze_backbone(model):
    is_resnet = 'resnet' in model.model_name
    for name, param in model.named_parameters():
        if ('fc' if is_resnet else 'head') in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

def unfreeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = True

def get_optimizer_groups(model, backbone_lr=1e-4, head_lr=1e-3, weight_decay=1e-4):
    head_names = {'fc', 'head'}
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(h in name for h in head_names):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return [
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': weight_decay},
        {'params': head_params, 'lr': head_lr, 'weight_decay': weight_decay}
    ]

print('Model builders and fine-tuning utilities successfully initialized.')


### 4. Training Engine with Two-Stage Fine-Tuning & Early Stopping


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_model(model_name, total_epochs=35, warmup_epochs=5, patience=8):
    print('=' * 75)
    print(f'STARTING TWO-STAGE TRAINING: {model_name.upper()}')
    print('=' * 75)

    model = (build_resnet50() if model_name == 'resnet50' else build_swin_t()).to(device)
    criterion = nn.CrossEntropyLoss()
    os.makedirs('checkpoints', exist_ok=True)
    best_ckpt = f'checkpoints/best_{model_name}.pth'

    # Stage 1: Warmup Classifier Head
    if warmup_epochs > 0:
        print(f'>>> Stage 1: Classifier Head Warmup ({warmup_epochs} epochs, frozen backbone) <<<')
        freeze_backbone(model)
        optimizer_warmup = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
        for ep in range(1, warmup_epochs + 1):
            model.train()
            t_loss, t_corr, t_tot = 0.0, 0, 0
            for imgs, targets in train_loader:
                imgs, targets = imgs.to(device), targets.to(device)
                optimizer_warmup.zero_grad()
                outs = model(imgs)
                loss = criterion(outs, targets)
                loss.backward()
                optimizer_warmup.step()
                t_loss += loss.item() * targets.size(0)
                t_corr += outs.argmax(1).eq(targets).sum().item()
                t_tot += targets.size(0)

            model.eval()
            v_loss, v_corr, v_tot = 0.0, 0, 0
            with torch.no_grad():
                for imgs, targets in val_loader:
                    imgs, targets = imgs.to(device), targets.to(device)
                    outs = model(imgs)
                    v_loss += criterion(outs, targets).item() * targets.size(0)
                    v_corr += outs.argmax(1).eq(targets).sum().item()
                    v_tot += targets.size(0)

            print(f'  [Warmup {ep:02d}/{warmup_epochs:02d}] Train Loss: {t_loss/t_tot:.4f}, Acc: {t_corr/t_tot*100:.1f}% || Val Loss: {v_loss/v_tot:.4f}, Acc: {v_corr/v_tot*100:.1f}%')

    # Stage 2: End-to-End Fine-Tuning
    print(f'>>> Stage 2: End-to-End Fine-Tuning ({total_epochs - warmup_epochs} epochs) <<<')
    unfreeze_backbone(model)
    optimizer = AdamW(get_optimizer_groups(model, backbone_lr=1e-4, head_lr=1e-3))
    scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=1e-6)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_val_acc = 0.0
    patience_cnt = 0

    for ep in range(1, total_epochs - warmup_epochs + 1):
        cur_ep = warmup_epochs + ep
        model.train()
        t_loss, t_corr, t_tot = 0.0, 0, 0
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            optimizer.zero_grad()
            outs = model(imgs)
            loss = criterion(outs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item() * targets.size(0)
            t_corr += outs.argmax(1).eq(targets).sum().item()
            t_tot += targets.size(0)
        scheduler.step()

        model.eval()
        v_loss, v_corr, v_tot = 0.0, 0, 0
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs, targets = imgs.to(device), targets.to(device)
                outs = model(imgs)
                v_loss += criterion(outs, targets).item() * targets.size(0)
                v_corr += outs.argmax(1).eq(targets).sum().item()
                v_tot += targets.size(0)

        tr_l, tr_a = t_loss / t_tot, (t_corr / t_tot) * 100.0
        va_l, va_a = v_loss / v_tot, (v_corr / v_tot) * 100.0
        history['train_loss'].append(tr_l)
        history['val_loss'].append(va_l)
        history['train_acc'].append(tr_a)
        history['val_acc'].append(va_a)

        mark = ''
        if va_l < best_val_loss:
            best_val_loss = va_l
            best_val_acc = va_a
            patience_cnt = 0
            torch.save({'state_dict': model.state_dict(), 'val_acc': best_val_acc}, best_ckpt)
            mark = ' [* BEST]'
        else:
            patience_cnt += 1

        print(f'  [Epoch {cur_ep:02d}/{total_epochs:02d}] Train Loss: {tr_l:.4f}, Acc: {tr_a:.1f}% || Val Loss: {va_l:.4f}, Acc: {va_a:.1f}%{mark}')
        if patience_cnt >= patience:
            print(f'Early stopping triggered after {patience} epochs without improvement.')
            break

    print(f'Finished {model_name}. Best Val Acc: {best_val_acc:.2f}% | Checkpoint saved: {best_ckpt}')
    return history


### 5. Training Execution: ResNet50 & Swin Transformer


In [ ]:
# Train ResNet50
history_resnet = train_model('resnet50', total_epochs=35, warmup_epochs=5, patience=8)

# Train Swin Transformer
history_swin = train_model('swin_t', total_epochs=35, warmup_epochs=5, patience=8)

# Plot Comparative Learning Curves
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history_resnet['val_loss'], 'b-o', label='ResNet50 Val Loss')
plt.plot(history_swin['val_loss'], 'r-s', label='Swin-T Val Loss')
plt.title('Validation Loss Comparison', fontweight='bold')
plt.xlabel('Fine-Tuning Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--')

plt.subplot(1, 2, 2)
plt.plot(history_resnet['val_acc'], 'b-o', label='ResNet50 Val Acc (%)')
plt.plot(history_swin['val_acc'], 'r-s', label='Swin-T Val Acc (%)')
plt.title('Validation Accuracy Comparison (%)', fontweight='bold')
plt.xlabel('Fine-Tuning Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, linestyle='--')

os.makedirs('outputs', exist_ok=True)
plt.tight_layout()
plt.savefig('outputs/training_curves_comparison.png', dpi=200)
plt.show()


### 6. Test Set Evaluation & Confusion Matrices (Held-Out Samples: 1, 4, 21)


In [ ]:
def evaluate_and_plot_cm(model_name):
    model = (build_resnet50(pretrained=False) if model_name == 'resnet50' else build_swin_t(pretrained=False)).to(device)
    ckpt = torch.load(f'checkpoints/best_{model_name}.pth', map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, targets in test_loader:
            imgs = imgs.to(device)
            outs = model(imgs)
            y_true.extend(targets.numpy())
            y_pred.extend(outs.argmax(1).cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    cm = np.zeros((4, 4), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    acc = np.trace(cm) / len(y_true) * 100.0

    # Plot CM
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, cmap='Blues')
    plt.colorbar(im)
    short_names = ['Very Healthy', 'Healthy', 'Spoiled', 'Very Spoiled']
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(short_names, rotation=20, ha='right')
    ax.set_yticklabels(short_names)
    ax.set_xlabel('Predicted Quality Grade', fontweight='bold')
    ax.set_ylabel('True Quality Grade', fontweight='bold')
    ax.set_title(f'{model_name.upper()} Test Set CM (Acc: {acc:.2f}%)', fontweight='bold')

    for i in range(4):
        for j in range(4):
            color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm[i, j]}\n({cm_norm[i, j]*100:.1f}%)', ha='center', va='center', color=color, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'outputs/confusion_matrix_{model_name}.png', dpi=200)
    plt.show()
    print(f'{model_name.upper()} Test Accuracy: {acc:.2f}%')

evaluate_and_plot_cm('resnet50')
evaluate_and_plot_cm('swin_t')


### 7. Grad-CAM Explainability: Defect Focus vs. Background Glare
Generates side-by-side attention maps proving whether ResNet50 and Swin Transformer focus on actual surface defects (browning & shrinkage).


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.acts = None
        self.grads = None
        target_layer.register_forward_hook(lambda m, inp, out: setattr(self, 'acts', out.detach()))
        target_layer.register_full_backward_hook(lambda m, gin, gout: setattr(self, 'grads', gout[0].detach()))

    def generate(self, input_tensor, target_class):
        self.model.eval()
        self.model.zero_grad()
        logits = self.model(input_tensor)
        score = logits[0, target_class]
        score.backward()

        acts, grads = self.acts, self.grads
        if acts.ndim == 4 and acts.shape[-1] in (768, 1024, 2048):
            acts = acts.permute(0, 3, 1, 2)
            grads = grads.permute(0, 3, 1, 2)

        w = torch.mean(grads, dim=(2, 3), keepdim=True)
        cam = torch.sum(w * acts, dim=1, keepdim=True)
        cam = torch.relu(cam).squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

# Initialize models and Grad-CAM
res_model = build_resnet50(pretrained=False).to(device)
res_model.load_state_dict(torch.load('checkpoints/best_resnet50.pth', map_location=device)['state_dict'])
res_cam = GradCAM(res_model, res_model.layer4[-1])

swin_model = build_swin_t(pretrained=False).to(device)
swin_model.load_state_dict(torch.load('checkpoints/best_swin_t.pth', map_location=device)['state_dict'])
swin_cam = GradCAM(swin_model, swin_model.features[7])

# Select representative test conditions
samples_to_show = [
    ('Control Day 0 (Fresh / Very Healthy)', DATASET_DIR / 'Control' / '0' / '1-b.jpg', 0),
    ('Control Day 7 (Browning Onset / Spoiled)', DATASET_DIR / 'Control' / '7' / '1-b.jpg', 2),
    ('1SKEO.2CEO Day 20 (Active Packaging Delay)', DATASET_DIR / '1SKEO.2CEO' / '20' / '1-b.jpg', 2),
    ('Control Day 20 (Severe Spoilage & Browning)', DATASET_DIR / 'Control' / '20' / '1-b.jpg', 3),
]

valid_show = [s for s in samples_to_show if Path(s[1]).exists()]
if valid_show:
    fig, axes = plt.subplots(len(valid_show), 3, figsize=(14, 4 * len(valid_show)))
    if len(valid_show) == 1:
        axes = np.expand_dims(axes, 0)

    for idx, (title, p, lbl) in enumerate(valid_show):
        img = Image.open(p).convert('RGB')
        t_in = eval_tf(img).unsqueeze(0).to(device)
        img_np = np.array(img.resize((224, 224)))

        h_res = res_cam.generate(t_in, lbl)
        h_swin = swin_cam.generate(t_in, lbl)

        hm_res_rgb = cv2.applyColorMap(np.uint8(255 * cv2.resize(h_res, (224, 224))), cv2.COLORMAP_JET)
        hm_swin_rgb = cv2.applyColorMap(np.uint8(255 * cv2.resize(h_swin, (224, 224))), cv2.COLORMAP_JET)

        ov_res = cv2.cvtColor(np.uint8(0.5 * img_np + 0.5 * cv2.cvtColor(hm_res_rgb, cv2.COLOR_BGR2RGB)), cv2.COLOR_RGB2BGR)
        ov_swin = cv2.cvtColor(np.uint8(0.5 * img_np + 0.5 * cv2.cvtColor(hm_swin_rgb, cv2.COLOR_BGR2RGB)), cv2.COLOR_RGB2BGR)

        axes[idx, 0].imshow(img_np)
        axes[idx, 0].set_title(f'{title}\nTarget: {CLASS_NAMES[lbl]}', fontweight='bold', fontsize=10)
        axes[idx, 0].axis('off')

        axes[idx, 1].imshow(cv2.cvtColor(ov_res, cv2.COLOR_BGR2RGB))
        axes[idx, 1].set_title('ResNet50 Heatmap (CNN)', fontweight='bold', fontsize=10, color='navy')
        axes[idx, 1].axis('off')

        axes[idx, 2].imshow(cv2.cvtColor(ov_swin, cv2.COLOR_BGR2RGB))
        axes[idx, 2].set_title('Swin-T Heatmap (Vision Transformer)', fontweight='bold', fontsize=10, color='darkred')
        axes[idx, 2].axis('off')

    plt.tight_layout()
    plt.savefig('outputs/gradcam_comparison.png', dpi=200)
    plt.show()
    print('Grad-CAM comparison successfully generated and saved to outputs/gradcam_comparison.png')
